### Data Modelling for processes Dataset

In [ ]:
import pandas as pd 

In [2]:
BASE_DATA_DIR = "../Dataset/processed_dataset.parquet"

In [3]:
df = pd.read_parquet(BASE_DATA_DIR)
df.head()

,flow iat mean,flow iat std,flow duration,packet length std,avg packet size,total fwd packets,down/up ratio,protocol,label
0,15714.666992,36508.757812,424296,420.939941,182.214279,17,0,6,Benign
1,101806.664062,176327.390625,305420,111.735405,150.750000,2,1,17,Benign
2,103997.000000,0.000000,103997,37.527767,107.500000,1,1,17,Benign
3,62.333332,66.665833,187,15.336231,77.750000,2,1,17,Benign
4,89694.000000,155310.406250,269082,55.319977,103.000000,2,1,17,Benign


In [4]:
df.count()

flow iat mean        100000
flow iat std         100000
flow duration        100000
packet length std    100000
avg packet size      100000
total fwd packets    100000
down/up ratio        100000
protocol             100000
label                100000
dtype: int64

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 9 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   flow iat mean      100000 non-null  float32
 1   flow iat std       100000 non-null  float32
 2   flow duration      100000 non-null  int32  
 3   packet length std  100000 non-null  float32
 4   avg packet size    100000 non-null  float32
 5   total fwd packets  100000 non-null  int32  
 6   down/up ratio      100000 non-null  int16  
 7   protocol           100000 non-null  int8   
 8   label              100000 non-null  object 
dtypes: float32(4), int16(1), int32(2), int8(1), object(1)
memory usage: 3.3+ MB


In [6]:
df.shape

(100000, 9)

#### Feature Engineering and Preprocessing

* We will calculate iat_cv, i.e. the coefficient of variation of inter-arrival times, as a feature to capture the variability in the time between events for each process.
* Formula is iat_cv = std(iat) / mean(iat)
* This will help us understand the burstiness of the process, which can be an important factor in rate limiting.

In [7]:
X = df.drop("label", axis=1)
y = df["label"]

In [8]:
EPSILON = 1e-6

X["iat_cv"] = df['flow iat std'] / (df['flow iat mean'] + EPSILON)

In [9]:
X = pd.get_dummies(X, columns=["protocol"], prefix="prot", drop_first=True)

In [10]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, LabelEncoder

scaler = MinMaxScaler(feature_range=(0, 1))

In [11]:
temp_scaled = scaler.fit_transform(X.drop(["prot_6", "prot_17"], axis=1))
X_scaled = pd.DataFrame(temp_scaled, columns=X.drop(["prot_6", "prot_17"], axis=1).columns)
X_scaled["prot_6"] = X["prot_6"]
X_scaled["prot_17"] = X["prot_17"]
X_scaled.head()

,flow iat mean,flow iat std,flow duration,packet length std,avg packet size,total fwd packets,down/up ratio,iat_cv,prot_6,prot_17
0,1.309639e-04,4.457724e-04,0.003536,0.088965,0.062402,0.000078,0.00000,0.040039,True,False
1,8.483972e-04,2.152960e-03,0.002545,0.023615,0.051627,0.000005,0.03125,0.029849,False,True
2,8.666500e-04,0.000000e+00,0.000867,0.007931,0.036815,0.000000,0.03125,0.000000,False,True
3,5.277778e-07,8.139906e-07,0.000002,0.003241,0.026627,0.000005,0.03125,0.018432,False,True
4,7.474583e-04,1.896342e-03,0.002242,0.011692,0.035274,0.000005,0.03125,0.029842,False,True


In [12]:
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)
y_encoded[:10]

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [13]:
final_data = pd.concat([X_scaled, pd.Series(y_encoded, name="label")], axis=1)
final_data.head()

,flow iat mean,flow iat std,flow duration,packet length std,avg packet size,total fwd packets,down/up ratio,iat_cv,prot_6,prot_17,label
0,1.309639e-04,4.457724e-04,0.003536,0.088965,0.062402,0.000078,0.00000,0.040039,True,False,0
1,8.483972e-04,2.152960e-03,0.002545,0.023615,0.051627,0.000005,0.03125,0.029849,False,True,0
2,8.666500e-04,0.000000e+00,0.000867,0.007931,0.036815,0.000000,0.03125,0.000000,False,True,0
3,5.277778e-07,8.139906e-07,0.000002,0.003241,0.026627,0.000005,0.03125,0.018432,False,True,0
4,7.474583e-04,1.896342e-03,0.002242,0.011692,0.035274,0.000005,0.03125,0.029842,False,True,0


In [16]:
final_data.to_parquet("../Dataset/final_data.parquet", index=False)